# B.8 依能力與成本，何時值得藉助工具？


直接回答可能比較快；用工具能得到可核對的結果，卻要多等一會。值不值得用，先量同一類題目兩條路的完整成功率，再定出答錯的成本與額外成本；不能把下一個 token 的高機率當成整題答對的把握。

紙上假設：直接回答的成功率 0.90，工具完整走完一輪的成功率 0.99；答錯一題的成本是 10 點，用工具另外多花 0.02 點。每題的平均成本是 `(1-成功率)×錯答成本+額外成本`。點數是人訂的尺度，不是秒或美元。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
direct_accuracy = 0.90
tool_accuracy = 0.99
wrong_cost = 10.0
tool_extra_cost = 0.02
direct_loss = (1 - direct_accuracy) * wrong_cost
tool_loss = (1 - tool_accuracy) * wrong_cost + tool_extra_cost
print("直接回答平均成本", round(direct_loss, 3))
print("工具流程平均成本", round(tool_loss, 3))
print("本例選擇", "TOOL" if tool_loss < direct_loss else "DIRECT")

直接回答的平均成本是 `(1-0.90)×10＝1.0`，工具流程是 `(1-0.99)×10+0.02＝0.12`，所以外層規則選 TOOL。程式裡的 `direct_loss`、`tool_loss` 存的就是這兩個平均成本，`round(..., 3)` 只是把小數縮短到三位。若直接回答的成功率改成 0.999，平均成本變成 0.01，就改選 DIRECT。這些數值都是假設，程式沒有估計信心、計時，也沒有訓練選卡模型。

工具的成功率要包括：選對工具與參數、真正執行、回填後的回答正確，不能只用函數計算無誤替整條路打分。前面工具算出 81、模型卻寫成 8 的錯例正說明這點。

要變成實際的決策規則，先用獨立的驗證題量兩條路的品質與等待時間，固定錯答成本與工具的額外成本，再依本節的算法：工具的平均成本比直接回答低，才選 TOOL。最後在沒有參與規則選擇的題目上檢查。改了問法、數字範圍或工具條件後，要重新檢查適用性；不能看最後哪題錯了才替那題另設門檻。這是固定決策規則的延伸，不是已經訓練出逐題判斷信心的能力。

練習只把 `tool_extra_cost` 改成 `1.5`，先預測：工具流程的平均成本變成 `0.1+1.5＝1.6`，比直接回答的 1.0 高，改選 DIRECT；再執行核對。等待的成本太高時，即使工具更準，也不一定值得用。
